# Baseline

## Imports

In [2]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import pandas as pd
import numpy as np
import sys
import warnings

project_dir = Path.cwd().parent

sys.path.insert(0, str(Path.cwd().parent)) # import src/
warnings.filterwarnings("ignore")

pd.set_option("display.width", 220, "display.max_columns", 30)

from src import (NestedCVRegressor, default_models,
                           default_param_spaces, MRMRSelector, run_per_family,
                           alt_param_spaces, load_per_family)

HVG_DATA_DIR = project_dir / 'data' / 'HVGs'

## Dataset

In [2]:
FAMILIES = {
    "KenyonCells":  (HVG_DATA_DIR / "Kenyon_cells_train_hvg.h5ad",  HVG_DATA_DIR / "Kenyon_cells_test_hng.h5ad"),
    "OpticLobe":    (HVG_DATA_DIR / "Optic_lobe_train_hvg.h5ad",    HVG_DATA_DIR / "Optic_lobe_test_hng.h5ad"),
    "Monoaminergic":(HVG_DATA_DIR / "Monoaminergic_train_hvg.h5ad",HVG_DATA_DIR / "Monoaminergic_test_hng.h5ad"),
    "Glia":         (HVG_DATA_DIR / "Glia_train_hvg.h5ad",         HVG_DATA_DIR / "Glia_test_hng.h5ad"),
    "Peptidergic":  (HVG_DATA_DIR / "Peptidergic_train_hvg.h5ad",  HVG_DATA_DIR / "Peptidergic_test_hng.h5ad"),
    "Clock":        (HVG_DATA_DIR / "Clock_train_hvg.h5ad",        HVG_DATA_DIR / "Clock_test_hng.h5ad"),
}
FAMILIES = {f: p for f, p in FAMILIES.items() if p[0].exists() and p[1].exists()}

## Baseline over all cell types

In [3]:
baseline = run_per_family(
    FAMILIES,
    models=default_models(),
    tune=False,
    dev="train",
    feature_selector=None,
    n_rounds=1,
    n_outer=5,
    keep_models=False,
    save_dir="../results/baseline",
    save_name="baseline",
)
NestedCVRegressor.format_table(baseline.summary)


=== Family: KenyonCells  (2278 cells, 2001 features) ===
[KenyonCells] Dummy: baseline ...
[KenyonCells] LinearRegression: baseline ...
[KenyonCells] ElasticNet: baseline ...
[KenyonCells] SVR: baseline ...
[KenyonCells] RandomForest: baseline ...
[KenyonCells] RandomForest_authors: baseline ...
[KenyonCells] XGBoost: baseline ...

=== Family: OpticLobe  (6365 cells, 2001 features) ===
[OpticLobe] Dummy: baseline ...
[OpticLobe] LinearRegression: baseline ...
[OpticLobe] ElasticNet: baseline ...
[OpticLobe] SVR: baseline ...
[OpticLobe] RandomForest: baseline ...
[OpticLobe] RandomForest_authors: baseline ...
[OpticLobe] XGBoost: baseline ...

=== Family: Monoaminergic  (748 cells, 2001 features) ===
[Monoaminergic] Dummy: baseline ...
[Monoaminergic] LinearRegression: baseline ...
[Monoaminergic] ElasticNet: baseline ...
[Monoaminergic] SVR: baseline ...
[Monoaminergic] RandomForest: baseline ...
[Monoaminergic] RandomForest_authors: baseline ...
[Monoaminergic] XGBoost: baseline ...

,model,MAE,RMSE,R2,Spearman
0,Dummy,6.304 (6.219–6.317),9.908 (9.652–9.911),-0.000 (-0.000–-0.000),nan (nan–nan)
1,LinearRegression,14.027 (12.991–14.215),18.093 (16.802–19.285),-2.470 (-2.786–-1.874),0.310 (0.264–0.360)
2,ElasticNet,3.942 (3.875–4.009),6.308 (6.220–6.336),0.591 (0.571–0.606),0.755 (0.727–0.791)
3,SVR,4.457 (4.441–4.516),9.218 (9.092–9.316),0.122 (0.113–0.134),0.804 (0.741–0.812)
4,RandomForest,2.693 (2.659–2.834),5.148 (4.931–5.419),0.730 (0.701–0.739),0.845 (0.831–0.857)
5,RandomForest_authors,4.102 (3.972–4.118),6.630 (6.280–6.753),0.553 (0.526–0.599),0.624 (0.553–0.724)
6,XGBoost,2.794 (2.490–3.124),5.095 (4.655–5.698),0.721 (0.670–0.779),0.841 (0.829–0.854)
7,Dummy,6.924 (6.912–6.936),10.621 (10.602–10.670),-0.000 (-0.000–-0.000),nan (nan–nan)
8,LinearRegression,5.333 (5.225–5.626),7.656 (7.613–9.069),0.478 (0.278–0.489),0.649 (0.623–0.662)
9,ElasticNet,4.084 (4.009–4.169),6.686 (6.583–6.811),0.604 (0.589–0.614),0.776 (0.762–0.798)


## Load results

In [3]:
baseline_per_fanily = load_per_family(outdir="../results/baseline", name="baseline")

## Best model per cell type

In [4]:
# best model per cell type by median MAE
best_per_type_mae = baseline_per_fanily.long.groupby(["family", "model"])["MAE"].median().reset_index()
best_per_type_mae = best_per_type_mae.loc[best_per_type_mae.groupby("family")["MAE"].idxmin()].sort_values("MAE")
display(best_per_type_mae)

,family,model,MAE
31,OpticLobe,RandomForest,2.688665
17,KenyonCells,RandomForest,2.693180
38,Peptidergic,RandomForest,4.335934
27,Monoaminergic,XGBoost,4.684793
10,Glia,RandomForest,5.798428
3,Clock,RandomForest,5.953855


In [5]:
# best model per cell type by median RMSE
best_per_type_rmse = baseline_per_fanily.long.groupby(["family", "model"])["RMSE"].median().reset_index()
best_per_type_rmse = best_per_type_rmse.loc[best_per_type_rmse.groupby("family")["RMSE"].idxmin()].sort_values("RMSE")
display(best_per_type_rmse)

,family,model,RMSE
34,OpticLobe,XGBoost,4.987506
20,KenyonCells,XGBoost,5.095239
38,Peptidergic,RandomForest,7.199889
24,Monoaminergic,RandomForest,7.789259
2,Clock,LinearRegression,8.474161
10,Glia,RandomForest,8.616459


In [ ]:
# best model per cell type by median RMSE, with MAE, R2, Pearson, Spearman (all with CIs)
best_per_type_rmse_ci = (
    baseline_per_fanily.summary
    .loc[baseline_per_fanily.summary.groupby("family")["RMSE_median"].idxmin()]
    .sort_values("RMSE_median")
    [["family", "model",
      "RMSE_median", "RMSE_lo", "RMSE_hi",
      "MAE_median",  "MAE_lo",  "MAE_hi",
      "R2_median",   "R2_lo",   "R2_hi",
      "Pearson_median", "Pearson_lo", "Pearson_hi",
      "Spearman_median", "Spearman_lo", "Spearman_hi"]]
)
# display(best_per_type_rmse_ci)

disp = best_per_type_rmse_ci.copy()
for m in ["RMSE", "MAE", "R2", "Pearson", "Spearman"]:
    disp[m] = disp.apply(
        lambda r: f"{r[f'{m}_median']:.3f} ({r[f'{m}_lo']:.3f}-{r[f'{m}_hi']:.3f})", axis=1)
disp = disp[["family", "model", "RMSE", "MAE", "R2", "Pearson", "Spearman"]]
display(disp)

,family,model,RMSE,MAE,R2,Pearson,Spearman
13,OpticLobe,XGBoost,4.988 (4.681–5.413),2.781 (2.642–2.848),0.779 (0.739–0.808),0.883 (0.861–0.899),0.870 (0.849–0.879)
6,KenyonCells,XGBoost,5.095 (4.655–5.698),2.794 (2.490–3.124),0.721 (0.670–0.779),0.849 (0.820–0.886),0.841 (0.829–0.854)
32,Peptidergic,RandomForest,7.200 (6.297–7.788),4.336 (4.008–4.791),0.670 (0.614–0.746),0.824 (0.787–0.888),0.784 (0.753–0.827)
18,Monoaminergic,RandomForest,7.789 (7.454–8.374),4.833 (4.647–5.000),0.637 (0.545–0.664),0.810 (0.758–0.839),0.819 (0.793–0.859)
36,Clock,LinearRegression,8.474 (8.439–8.840),6.264 (6.136–6.489),0.575 (0.541–0.613),0.759 (0.739–0.787),0.653 (0.642–0.808)
25,Glia,RandomForest,8.616 (7.913–9.028),5.798 (5.549–6.112),0.800 (0.780–0.832),0.898 (0.887–0.916),0.913 (0.901–0.917)
